# Lensless Imaging — Demo

This notebook shows how to use the repository end to end:

1. Clone the repository and install dependencies.
2. Download the model checkpoints.
3. Download a custom `.zip` dataset (Google Drive link), run `inference.py` on it, and save reconstructions.
4. Visualize a few samples (original vs. lensless measurement vs. reconstruction).
5. Run `calculate_metrics.py` to print PSNR / SSIM / LPIPS / MSE (if ground truth is available).

It is designed to run in a fresh Google Colab session. Just set the two links below and run all cells.

## 1. Clone the repository and install dependencies

In [ ]:
# Clone the repository (replace with your public repository URL).
REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"

!git clone $REPO_URL repo
%cd repo

In [ ]:
# Install the required packages.
!pip install -q -r requirements.txt

**Note:** if `pip` downgraded `numpy`, Colab may ask you to restart the runtime. After restarting, re-run the `%cd repo` cell and continue from here (do not re-run the clone cell).

## 2. Download the checkpoints

The checkpoints are downloaded from Google Drive (the file ids are configured in `download_checkpoints.py`). We use the modular pre+post model in this demo.

In [ ]:
MODEL = "modular_pre_post"  # one of: modular_pre_post, modular_pre, modular_post, unrolled_admm20

!python download_checkpoints.py --model $MODEL

## 3. Download the dataset and run inference

Paste a Google Drive link to a `.zip` dataset below. The archive must unzip to a directory with the following structure:

```
data
├── lensless           # lensless measurements (required)
│   └── ID.png
├── masks              # mask patterns (required)
│   └── ID.npy
└── lensed             # ground-truth images (optional)
    └── ID.png
```

In [ ]:
import os
import zipfile
from pathlib import Path

import gdown

# Google Drive link to a .zip dataset (replace with your link).
DATA_URL = "https://drive.google.com/file/d/PUT_YOUR_FILE_ID_HERE/view?usp=sharing"

# Download and unzip the dataset.
gdown.download(DATA_URL, "dataset.zip", quiet=False, fuzzy=True)
with zipfile.ZipFile("dataset.zip", "r") as archive:
    archive.extractall("data_custom")

# Find the directory that contains the 'lensless' folder.
DATA_DIR = next(p.parent for p in Path("data_custom").rglob("lensless") if p.is_dir())
RECON_DIR = Path("reconstructions").absolute()
print("Data directory:", DATA_DIR)
print("Reconstructions will be saved to:", RECON_DIR)

In [ ]:
# Run inference on the custom dataset and save reconstructions.
!python inference.py \
    model=$MODEL \
    inferencer.from_pretrained=saved/$MODEL/model_best.pth \
    datasets=custom_dir \
    datasets.test.data_dir="$DATA_DIR" \
    inferencer.save_path="$RECON_DIR"

## 4. Visualize some samples

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

recon_paths = sorted(RECON_DIR.glob("*.png"))[:4]
lensless_dir = DATA_DIR / "lensless"
lensed_dir = DATA_DIR / "lensed"

n = len(recon_paths)
fig, axes = plt.subplots(n, 3, figsize=(9, 3 * n))
axes = np.atleast_2d(axes)
for row, recon_path in enumerate(recon_paths):
    sample_id = recon_path.stem

    lensless = np.asarray(Image.open(lensless_dir / f"{sample_id}.png").convert("RGB")).astype(np.float32)
    lensless = lensless / lensless.max()  # contrast stretch for visualization
    reconstruction = np.asarray(Image.open(recon_path).convert("RGB"))

    lensed_path = lensed_dir / f"{sample_id}.png"
    if lensed_path.exists():
        axes[row, 0].imshow(Image.open(lensed_path).convert("RGB"))
    axes[row, 0].set_title("original (lensed)")
    axes[row, 1].imshow(lensless)
    axes[row, 1].set_title("lensless measurement")
    axes[row, 2].imshow(reconstruction)
    axes[row, 2].set_title("reconstruction")
    for ax in axes[row]:
        ax.axis("off")
plt.tight_layout()
plt.show()

## 5. Compute metrics

If the dataset contains ground-truth `lensed` images, compute PSNR / SSIM / LPIPS / MSE between the reconstructions and the ground truth.

In [ ]:
!python calculate_metrics.py \
    --ground_truth "$DATA_DIR" \
    --reconstructions "$RECON_DIR"